# 📘 Annotated edition · 02_load_parallel_validate

A copy of `notebooks/02_load_parallel_validate.ipynb`. Original cells and saved outputs are unchanged. New **📘 Explanation** and **✅ Answer** cells are added.

**The point of this notebook:** read **all 108 booking files** from two different systems into **one table at one grain**, then pass every booking through a **contract** (a pydantic model). Every booking is either accepted or rejected with a reason, and *input = accepted + rejected* is checked.

**Key results from the saved run:**
- legacy: **1,252,765** booking-test rows from 102 files, **21** repeated header lines dropped. nova: **62,418** rows from 6 files.
- combined: **1,315,183** booking-test rows, **567,986** distinct bookings.
- validation: **564,879 accepted + 3,107 rejected = 567,986** ✔ (all rejects are legacy).
- booking-grain table: 564,879 rows, including **1,500 nova reschedule records**.

> ℹ️ **One line changed in this annotated copy:** `ROOT = ... if pathlib.Path.cwd().name in ("notebooks", "notebook_new") else ...`. The original only recognised a folder called `notebooks`. Run from `notebook_new/`, it would have looked for `notebook_new/data/raw` and failed. Nothing else in the code was changed.

# 02 · Load everything in parallel, validate every record at the boundary

**Produces:** `data/clean/bookings_all.csv.gz` (one row per booking-test, both systems, normalised), `data/clean/bookings.csv.gz` (one row per booking), `data/clean/rejects_bookings.csv` (every rejected record with its reason).

**Needs:** notebook 01 run; the pack in `data/raw/`.

Two ideas in this notebook. First, the pack is a few hundred files, so the unit of work is a file and a process pool reads them side by side. Second, nothing enters the analysis without passing a model that says what a booking must look like; what fails is kept, counted and listed, so input equals accepted plus rejected.

#### 📘 Explanation: header

Three output files, each at a stated **grain** (what one row represents):
- `bookings_all.csv.gz`: **one row per booking-test**, both systems, normalised columns. The detailed table.
- `bookings.csv.gz`: **one row per booking**, with `n_tests` and `amount_rs`. The table the headline is computed from.
- `rejects_bookings.csv`: **every rejected booking with its reason**. The evidence that nothing was silently dropped.

**"The unit of work is a file":** you can't parallelise *inside* one gzip stream easily, but you can read 108 files side by side.

## CONFIG

#### 📘 Explanation: the CONFIG cell, line by line

This cell is the same in notebooks 01–06. Explained once in full:

| Code | What it does and why |
|---|---|
| `import sys, pathlib, time` | `sys` to edit the import path, `pathlib` for file paths that work on Windows and Linux, `time` to time steps. |
| `ROOT = pathlib.Path.cwd().parent if ... == "notebooks" else pathlib.Path.cwd()` | Finds the **project root**. Jupyter usually starts inside the notebook's folder, so the root is one level up. When run from the root itself (for example a script), `cwd()` is already the root. Every path below is built from `ROOT`, so the notebook works on any machine. |
| `sys.path.insert(0, str(ROOT / "src"))` | Puts `src/` at the front of Python's import search path, so `from kalpa import ...` finds the project's own package without `pip install`. |
| `import numpy as np` / `import pandas as pd` | The two core libraries: arrays and fast maths / tables. |
| `from kalpa import loaders, models, checks, logs` | The shared project package: **loaders** read the raw files, **models** are pydantic "contracts" for a valid record, **checks** print PASS/FAIL, **logs** append rows to the markdown logs in `docs/`. |
| `pd.set_option("display.max_columns", 40)` / `("display.width", 160)` | Print wide tables in full instead of truncating them with `...`. |
| `CONFIG = {...}` | All run-specific settings in one place (see notebook 00): brief, group, `raw` and `clean` folders as `pathlib` paths, the two half-years being compared, the seed, and `workers` (`None` means use every CPU core in the process pool). |
| `CONFIG["clean"].mkdir(parents=True, exist_ok=True)` | Creates `data/clean/` if it doesn't exist, and doesn't fail if it does. A fresh clone can run straight away. |
| `rng = np.random.default_rng(CONFIG["seed"])` | A seeded random generator **object**. Anything random (the shuffle test in 05) uses `rng`, so results are reproducible. |
| `print(pd.__version__, np.__version__)` | Prints library versions (**2.3.3** and **2.3.5** in the saved run). If a re-run elsewhere gives different numbers, a version difference is the first suspect. pandas 3.x, for example, changed the default string dtype. |

In [2]:
import sys, pathlib, time
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name in ("notebooks", "notebook_new") else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import numpy as np
import pandas as pd
from kalpa import loaders, models, checks, logs
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
CONFIG = {
    "brief": "<LETTER>",
    "group": "<N>",
    "raw": ROOT / "data" / "raw",
    "clean": ROOT / "data" / "clean",
    "period_this": ("2026-01-01", "2026-06-30"),
    "period_last": ("2025-01-01", "2025-06-30"),
    "seed": 20261019,
    "workers": None,          # None = all cores
}
CONFIG["clean"].mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(CONFIG["seed"])
print(pd.__version__, np.__version__)

2.3.3 2.3.5


## 1 · MAP: two systems, one grain

Legacy exports are one row per booking-test. Nova exports are one JSON object per booking with the tests nested. Both are brought to booking-test grain by the loaders, then rolled up to booking grain here. The picture:

```
legacy/*.csv.gz  ─┐                           ┌─ bookings_all (booking-test grain)
                  ├─ process pool ─ concat ───┤
nova/*.jsonl.gz  ─┘        │                  └─ bookings (booking grain, n_tests, amount)
                           └─ pydantic validation ─ rejects_bookings
```

#### 📘 Explanation: MAP, two systems and one grain

- **Legacy** CSV already has one row per test (booking-test grain).
- **Nova** JSONL has one JSON object per booking, with its tests nested in a list: `{"booking_id": ..., "tests": [{"code": "T016", "price": 160}, ...]}`. `loaders.read_nova_file` **flattens** it into one row per test so it has the same grain as legacy.
- Both are concatenated, then **rolled up** to booking grain.
- The validator branch produces the rejects file.

Why bring nova *down* to test grain and then back *up*? Because legacy can only be read at test grain. The common denominator is the finer grain, and from there one roll-up function serves both systems.

## 2 · DO: sequential against parallel, timed, so the choice is a measured one

#### 📘 Explanation
"Measured, not assumed": before committing to parallel code, time it against the simple version on the same input.

#### 📘 Explanation: sequential vs process pool

- `legacy_files` / `nova_files`: sorted lists of every file of each kind. Sorting makes row order deterministic.
- **Sequential:** `pd.concat([loaders.read_legacy_file(p) for p in legacy_files[:12]], ignore_index=True)` reads 12 files one after another and stacks them. `ignore_index=True` renumbers rows 0..n-1 instead of repeating each file's 0..k.
- **Parallel:** `loaders.read_many(legacy_files[:12], "legacy", workers=...)` uses a `ProcessPoolExecutor`. It starts worker processes, sends each a `(kind, path)` job, and collects the DataFrames **back into their original order** (`frames[futures[fut]] = fut.result()`), so the output is identical to sequential.
- `len(seq) == len(par)` is a correctness check: the fast way must give the same rows.

In [3]:
legacy_files = sorted((CONFIG["raw"] / "bookings" / "legacy").rglob("*.csv.gz"))
nova_files = sorted((CONFIG["raw"] / "bookings" / "nova").rglob("*.jsonl.gz"))
print(len(legacy_files), "legacy files,", len(nova_files), "nova files")

t0 = time.time()
seq = pd.concat([loaders.read_legacy_file(p) for p in legacy_files[:12]], ignore_index=True)
t_seq = time.time() - t0
t0 = time.time()
par = loaders.read_many(legacy_files[:12], "legacy", workers=CONFIG["workers"])
t_par = time.time() - t0
print(f"12 files: sequential {t_seq:.1f}s, process pool {t_par:.1f}s, same rows: {len(seq) == len(par)}")

102 legacy files, 6 nova files
12 files: sequential 1.4s, process pool 3.4s, same rows: True


#### 📘 Reading the output, and a surprise

**12 files: sequential 1.4 s, process pool 3.4 s.** The pool was *slower* here. That's a real, honest result:
- On **Windows**, worker processes start by **spawning** a fresh Python interpreter that re-imports pandas, which takes ~1 s per worker.
- Each result DataFrame is **pickled** and copied back to the main process.
- With only 12 small files (~0.1 s each), that overhead outweighs the saving.

The pool pays off when the work per file is large relative to the start-up cost: the full load below reads 102 files in 9.8 s. The lesson for the approach log is that parallelism has a fixed cost, so measure it at the real scale. A fairer comparison would time all 102 files both ways.

> **Interview angle.** Why a process pool and not a thread pool here? When would threads be the right choice?
>
> *What a good answer covers:* Decompressing gzip, parsing CSV and parsing JSON are CPU-bound Python work, and the global interpreter lock lets only one thread run Python bytecode at a time, so threads would queue. Processes each have their own interpreter. Threads win when the work waits on I/O, such as network calls or disk reads of already-parsed data, and when the data to pass between workers is large, since processes copy it.

#### ✅ Answer: why a process pool and not a thread pool? When would threads be right?

**Why processes here:** the work per file is **CPU-bound Python**: gzip decompression, splitting lines, parsing CSV text into numbers and dates, `json.loads` per line for nova. CPython has a **Global Interpreter Lock (GIL)**: only one thread can execute Python bytecode at a time in a process. With a thread pool the threads would mostly take turns, giving close to sequential speed plus switching overhead. (Some C code, such as parts of pandas' C parser and zlib, releases the GIL, but the line filtering and JSON parsing in our loaders is pure Python.) **Each process has its own interpreter and its own GIL**, so N processes can use N cores truly in parallel.

**The costs of processes:** start-up (spawn on Windows/macOS re-imports modules), and **pickling**: arguments and results are serialised and copied between processes. That's why `_dispatch` is a top-level function (only top-level functions can be pickled) and why we send *paths*, not data, to the workers.

**When threads are the right choice:**
1. **I/O-bound work**: downloading files, calling an API, querying a database, reading from a network share. The thread spends most of its time *waiting*, and waiting releases the GIL, so threads overlap the waits cheaply.
2. **Large shared data**: threads share memory, so there's nothing to copy. Processes would have to pickle a large DataFrame to every worker.
3. **Work inside libraries that release the GIL** (NumPy maths on big arrays, some compression), where threads get real parallelism without the copying cost.
4. Many small, quick tasks, where process start-up would dominate (as in the 12-file test above).

(Free-threaded Python 3.13t/3.14t removes the GIL as an option, but the standard build used here still has it.)

## 3 · DO: the full load

#### 📘 Explanation: the full load

- `loaders.read_many(legacy_files, "legacy", workers=CONFIG["workers"])` reads all 102 legacy files in parallel. `workers=None` means as many processes as CPU cores.
- `legacy.attrs['repeated_header_lines']`: `read_many` sums each file's count into the combined frame's `attrs`.
- The same for nova. `if nova_files else pd.DataFrame()` guards against a pack with no nova files (for example the sample pack).

In [4]:
t0 = time.time()
legacy = loaders.read_many(legacy_files, "legacy", workers=CONFIG["workers"])
print(f"legacy: {len(legacy):,} booking-test rows from {len(legacy_files)} files in {time.time()-t0:.1f}s; repeated headers dropped: {legacy.attrs['repeated_header_lines']}")
t0 = time.time()
nova = loaders.read_many(nova_files, "nova", workers=CONFIG["workers"]) if nova_files else pd.DataFrame()
print(f"nova: {len(nova):,} booking-test rows from {len(nova_files)} files in {time.time()-t0:.1f}s")

legacy: 1,252,765 booking-test rows from 102 files in 9.8s; repeated headers dropped: 21
nova: 62,418 booking-test rows from 6 files in 4.9s


#### 📘 Reading the output

- **legacy: 1,252,765 rows from 102 files in 9.8 s, 21 repeated headers dropped.** The 0 found in the single file in notebook 01 didn't mean "none exist", which is why you profile the whole pack.
- **nova: 62,418 rows from 6 files in 4.9 s.** Nova is slower per row because each line needs `json.loads`, plus a Python loop over nested tests.

#### 📘 Explanation: combining two frames whose columns differ

- `set(legacy.columns) - set(nova.columns)`: columns only in legacy. The reverse gives columns only in nova. These are printed **before** concatenating so a missing column can't surprise you as an all-NaN column later.
- `for c in only_nova: legacy[c] = pd.NA` adds nova-only columns to legacy, filled with missing values (and the reverse).
- `pd.concat([legacy, nova[legacy.columns]], ignore_index=True)` stacks them, with nova's columns put **in legacy's order**.
- `all_rows["city"].astype("string").replace({"Bangalore": "Bengaluru"})`: the legacy vendor spells Bengaluru two ways. One business city should have one name, otherwise a groupby by city splits Bengaluru into two rows. `.astype("string")` first, because replacing values in a `category` column with categories that differ between the two frames is awkward.
- The `groupby("system", observed=True).agg(rows=..., bookings=("booking_id","nunique"), first=("slot_at","min"), last=("slot_at","max"))` summary per system.

In [6]:
# Concatenating two frames whose columns differ: name the difference before you union, so a silent NaN column cannot surprise you later
only_legacy = set(legacy.columns) - set(nova.columns)
only_nova = set(nova.columns) - set(legacy.columns)
print("only in legacy:", only_legacy, "| only in nova:", only_nova)
for c in only_nova: legacy[c] = pd.NA
for c in only_legacy: nova[c] = pd.NA
all_rows = pd.concat([legacy, nova[legacy.columns]], ignore_index=True)
all_rows["city"] = all_rows["city"].astype("string").replace({"Bangalore": "Bengaluru"})
print(f"{len(all_rows):,} booking-test rows; {all_rows['booking_id'].nunique():,} distinct bookings")
all_rows.groupby("system", observed=True).agg(rows=("booking_id", "size"), bookings=("booking_id", "nunique"), first=("slot_at", "min"), last=("slot_at", "max"))

only in legacy: set() | only in nova: {'status_at_export', 'rescheduled_from'}


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_12248\2537870784.py:7: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_rows = pd.concat([legacy, nova[legacy.columns]], ignore_index=True)


1,315,183 booking-test rows; 567,986 distinct bookings


,rows,bookings,first,last
system,,,,
legacy,1252765,541464,2025-01-01 06:00:00,2026-06-30 19:59:00
nova,62418,26522,2026-04-01 06:01:00,2026-07-04 10:50:00


#### 📘 Reading the output

- **only in nova: `status_at_export`, `rescheduled_from`.** Nova records reschedules explicitly (a new booking pointing at the old one, with the old one marked `rescheduled`). Legacy has no such fields because it **overwrote the slot in place**. That difference is the identity problem notebook 03 solves.
- **FutureWarning** about concatenating empty or all-NA columns: pandas is warning that in a future version an all-NA column (the `pd.NA` columns we just added to legacy) will affect the result dtype. Harmless here. Silence it by creating those columns with an explicit dtype, for example `legacy[c] = pd.Series(pd.NA, index=legacy.index, dtype="string")`.
- **1,315,183 booking-test rows, 567,986 distinct bookings.**
- Per system: legacy 1,252,765 rows / 541,464 bookings (**2.31** tests per booking), slots 2025-01-01 to 2026-06-30. Nova 62,418 / 26,522 (**2.35**), slots **2026-04-01 to 2026-07-04**. Nova contains a few bookings slotted **after** 30 June (exported early July), which fall outside H1 2026.

### SEE: the grain trap, in numbers

The table above is the whole of brief B in one line: rows per booking differ by system, so a row count across the migration is not a booking count. Nova also carries rescheduled records. Decide now what a booking is, write the rule, and apply it in notebook 03.

#### 📘 Explanation: SEE, the grain trap in numbers

- Tests per booking differ by system (2.31 vs 2.35), so **row counts aren't booking counts**, and the *ratio* shifts when Chennai and Jaipur move to nova. A city's "rows" can change with no change in bookings.
- Nova also contains **rescheduled records**: one real visit can appear as *two* booking ids. Counting distinct ids would count that visit twice in nova, but only once in legacy.

"Decide now what a booking is" (answered at the end of this notebook): **one visit by one patient**, and a rescheduled pair is one visit.

## 4 · DO: validate every booking with a model

`models.Booking` says what one visit must look like. Build one record per booking from the rows, validate each, and keep the rejects with reasons. This is slower than a vectorised check and that is the point of doing it once at the boundary: the model is the contract, and the rejects file is the evidence that nothing slipped past it.

#### 📘 Explanation: validating with a model

`models.Booking` (in `src/kalpa/models.py`) is a **pydantic** class describing a valid booking:
- `booking_id` must match the legacy **or** nova pattern.
- `system` must be "legacy" or "nova". `channel` must be "home" or "clinic". `source` must be "app", "phone" or "walkin". Values are lower-cased first.
- `city`: "Bangalore" is mapped to "Bengaluru".
- `tests: list[TestLine]` with **at least 1** test. Each test code must match `T###` or `PK##`, and price must be ≥ 0.
- a model-level rule: **`slot_at` must not be before `created_at`** (you can't attend before you book).
- strings are stripped of whitespace automatically (`str_strip_whitespace=True`).

It's slower than vectorised pandas checks (a Python object per booking), so it runs **once, at the boundary** between raw and clean data. After that everything is vectorised.

#### 📘 Explanation: `records_from_rows`, a fast group-by without `apply`

Goal: turn 1.3 million booking-test rows into 567,986 dicts, one per booking with a nested `tests` list, in the shape the model expects.

1. `r = rows.sort_values("booking_id", kind="stable")`: sorting puts all rows of a booking next to each other. `stable` keeps their original order within a booking.
2. `ids = r["booking_id"].to_numpy()` gives a plain NumPy array, which is fast to index.
3. `_, starts = np.unique(ids, return_index=True)`: for the sorted ids, the index where each distinct id **first appears**. That's the start of each booking's block.
4. `ends = np.append(starts[1:], len(r))`: each block ends where the next begins, and the last ends at the final row.
5. `meta = r[cols].to_dict("records")`: booking-level fields as a list of dicts, built **once** for all rows.
6. `codes`, `prices`: NumPy arrays of test codes and prices. Missing code becomes `""`, missing price becomes 0.
7. `for a, b in zip(starts, ends)`: for each booking, take booking fields from its first row (`meta[a]`), converting `pd.NA`/NaN to `None` (pydantic understands `None` as missing, not `pd.NA`), and build `tests` from `codes[a:b]` and `prices[a:b]`.
8. `yield rec` makes it a **generator**: records are produced one at a time and never all held in memory at once.

**Why not `rows.groupby("booking_id").apply(lambda g: ...)`?** `apply` calls a Python function **per group**, building a small DataFrame each time. 567,986 small DataFrames take minutes. Sorting once and slicing NumPy arrays takes seconds and gives the same result.

Then:
- `models.validate_many(models.Booking, records_from_rows(all_rows))` tries `Booking.model_validate(rec)` on each record. Successes go to `accepted`. Failures go to `rejects` as `{"record": ..., "errors": [(field, message), ...]}`. It never raises.
- `checks.reconciles(...)` checks **distinct bookings in = accepted + rejected**.

In [7]:
def records_from_rows(rows: pd.DataFrame):
    """Group booking-test rows into one dict per booking, the shape the model expects.

    Sorted once, then sliced by the boundaries np.unique returns: a few seconds for a million rows,
    where a groupby.apply with a Python lambda would take minutes. Same result, measured difference."""
    cols = ["booking_id", "system", "created_at", "slot_at", "city", "clinic_id", "channel", "source", "patient_id", "home_fee", "rescheduled_from", "status_at_export"]
    r = rows.sort_values("booking_id", kind="stable").reset_index(drop=True)
    ids = r["booking_id"].to_numpy()
    _, starts = np.unique(ids, return_index=True)
    ends = np.append(starts[1:], len(r))
    meta = r[cols].to_dict("records")
    codes = r["test_code"].astype("string").fillna("").to_numpy()
    prices = r["test_price"].fillna(0).astype(int).to_numpy()
    for a, b in zip(starts, ends):
        rec = {k: (None if (v is pd.NA or (isinstance(v, float) and v != v)) else v) for k, v in meta[a].items()}
        rec["tests"] = [{"code": c, "price": int(p)} for c, p in zip(codes[a:b], prices[a:b])]
        yield rec

t0 = time.time()
accepted, rejects = models.validate_many(models.Booking, records_from_rows(all_rows))
print(f"validated {len(accepted)+len(rejects):,} bookings in {time.time()-t0:.0f}s (Python-level validation, once, at the boundary): accepted {len(accepted):,}, rejected {len(rejects):,}")
checks.reconciles("booking validation", all_rows["booking_id"].nunique(), len(accepted), len(rejects))

validated 567,986 bookings in 44s (Python-level validation, once, at the boundary): accepted 564,879, rejected 3,107
PASS  booking validation: input = kept + rejected   567,986 = 564,879 + 3,107


True

#### 📘 Reading the output

- **567,986 bookings validated in 44 s: 564,879 accepted, 3,107 rejected.** 44 s is fine because it runs once.
- **PASS: 567,986 = 564,879 + 3,107.** No booking vanished.

#### 📘 Explanation: the rejects file

- Builds one row per reject: `booking_id`, `system`, and `reason`, which joins every `field: message` pair with `; `.
- Saves it to `data/clean/rejects_bookings.csv`. This is the audit evidence.
- `rej["reason"].str.split(":").str[0].value_counts()` takes the text before the first `:` (the **field of the first error**) and counts.

In [8]:
rej = pd.DataFrame([{"booking_id": r["record"].get("booking_id"), "system": r["record"].get("system"),
                     "reason": "; ".join(f"{f}: {m}" for f, m in r["errors"])} for r in rejects])
rej.to_csv(CONFIG["clean"] / "rejects_bookings.csv", index=False)
rej["reason"].str.split(":").str[0].value_counts() if len(rej) else print("no rejects")

reason
model           1027
booking_id       793
tests.0.code     583
tests.1.code     378
tests.2.code     222
tests.3.code      80
tests.4.code      18
tests.5.code       6
Name: count, dtype: int64

#### 📘 Reading the output: what the 3,107 rejects actually are

| First failing field | Count | What it is (checked in the rejects file) |
|---|---|---|
| `model` | 1,027 | "slot ... is before creation ...". In **every one** of the 1,027, creation is **exactly 2 days** after the slot, to the minute. That's a systematic vendor error on `created_at`, not random bad data. |
| `booking_id` | 793 | the 7-digit ids from notebook 01, **all ending in an extra `0`**. Trimmed, none collides with another booking. |
| `tests.0.code` ... `tests.5.code` | 583 + 378 + 222 + 80 + 18 + 6 = 1,287 | an **empty test code** at position N of the booking's test list (`tests.2.code` = the 3rd test) |

Total 3,107. **All rejects are legacy**: nova's records passed. They're spread evenly across periods: 1,074 rejected bookings fall in H1 2025 and 1,004 in H1 2026.

> **Interview angle.** What does a pydantic model give you that a pandas `.str.match` does not, and what does it cost?
>
> *What a good answer covers:* A single, named contract for a record that normalises what is safe to normalise and refuses the rest with a per-field reason, reusable by the SQL loader, the API and the tests. It costs time per record (Python-level validation) and a second representation of the data, so you run it once at the boundary and vectorise everything after.

#### ✅ Answer: what does a pydantic model give you that `.str.match` doesn't, and what does it cost?

**What it gives:**
1. **One named contract for a whole record.** `.str.match` checks one column. The model checks every field *and* rules across fields (slot not before creation, at least one test) in one place called `Booking`.
2. **Safe normalisation alongside refusal.** It lower-cases `channel`, strips whitespace and maps "Bangalore" to "Bengaluru" (things that are safe to fix), and refuses things that aren't (malformed ids, empty test codes).
3. **A per-field reason for every rejection** (`tests.2.code: test code '' does not match T### or PK##`). That gives a rejects file that explains itself, which is exactly what an auditor wants.
4. **Nested structure**: a booking *with a list of tests*, each validated as a `TestLine`. Hard to express with column-wise string checks.
5. **Reuse**: the same class can validate records in the SQL loader, an API endpoint or a unit test, so the definition of "a valid booking" exists **once**.
6. **Typed outputs**: accepted objects have real `datetime` and `int` fields, plus derived properties (`n_tests`, `amount_rs`).

**What it costs:**
1. **Speed**: it builds a Python object per record, so 567,986 bookings take 44 s, where a vectorised `.str.match` over the same column takes well under a second.
2. **Memory and a second representation**: the data exists as a DataFrame *and* as model objects, and you convert back (`model_dump`).
3. **Code to maintain**: the model must evolve with the sources.

**So** run it **once at the boundary** where raw data enters, and use vectorised pandas for everything after.

## 5 · DO: the booking-grain table

#### 📘 Explanation: building the booking-grain table

- For each accepted `Booking` object `b`:
  - `b.model_dump(exclude={"tests"})` gives the booking fields as a dict, without the nested tests list.
  - `"n_tests": b.n_tests` (number of tests), `"amount_rs": b.amount_rs` (sum of test prices **plus home_fee**).
  - `"has_package"`: whether any test code starts with `PK` (a package of tests).
- `pd.to_datetime(...).dt.tz_localize(None)` makes the dates plain, timezone-naive timestamps. The loaders already converted nova from UTC to IST, so both systems are IST.
- `month = slot_at.dt.to_period("M").astype(str)` gives `"2026-03"` style labels for monthly grouping (used in SQL and the plan join).
- `is_reschedule = rescheduled_from.notna()`: True for nova records that **replace** an earlier booking.
- `groupby(["system","is_reschedule"]).size()` counts each combination.

In [9]:
bookings = pd.DataFrame([{**b.model_dump(exclude={"tests"}), "n_tests": b.n_tests, "amount_rs": b.amount_rs,
                          "has_package": any(t.code.startswith("PK") for t in b.tests)} for b in accepted])
for c in ("created_at", "slot_at"):
    bookings[c] = pd.to_datetime(bookings[c]).dt.tz_localize(None)
bookings["month"] = bookings["slot_at"].dt.to_period("M").astype(str)
bookings["is_reschedule"] = bookings["rescheduled_from"].notna()
print(bookings.shape)
bookings.groupby(["system", "is_reschedule"]).size()

(564879, 17)


system  is_reschedule
legacy  False            538357
nova    False             25022
        True               1500
dtype: int64

#### 📘 Reading the output

- `(564879, 17)`: one row per accepted booking, 17 columns.
- legacy, not reschedule: **538,357**. nova, not reschedule: **25,022**. nova, reschedule: **1,500**.
- Legacy has no reschedule rows at all, because legacy overwrote the slot in place. The 1,500 nova reschedule records point (`rescheduled_from`) to originals that are still in the table, marked `status_at_export = "rescheduled"`. There are 1,499 such originals, and notebook 03 drops them. One reschedule's original isn't marked, so check that one by hand.

#### 📘 Explanation: save and check

- `all_rows.to_csv(... "bookings_all.csv.gz", compression="gzip")`: the booking-test table, including rejected bookings' rows. It's the *raw-but-normalised* layer, so rejects can still be looked up later.
- `bookings.to_csv(... "bookings.csv.gz")`: the booking-grain table, accepted bookings only.
- Check 1: `bookings["booking_id"].is_unique`, the grain is truly one row per booking.
- Check 2: every booking has at least one test (guaranteed by the model's `min_length=1`, re-checked on the output).
- `checks.summary()` gives all checks run in this notebook.

In [10]:
all_rows.to_csv(CONFIG["clean"] / "bookings_all.csv.gz", index=False, compression="gzip")
bookings.to_csv(CONFIG["clean"] / "bookings.csv.gz", index=False, compression="gzip")
checks.check("booking table has one row per booking id", bookings["booking_id"].is_unique, len(bookings))
checks.check("n_tests per booking is at least 1", bool((bookings["n_tests"] >= 1).all()))
checks.summary()

PASS  booking table has one row per booking id   564879
PASS  n_tests per booking is at least 1
3 of 3 checks passed


,check,passed,detail
0,booking validation: input = kept + rejected,True,"567,986 = 564,879 + 3,107"
1,booking table has one row per booking id,True,564879
2,n_tests per booking is at least 1,True,


#### 📘 Reading the output
3 of 3 checks passed: the validation reconciles, booking ids are unique, and every booking has at least one test.

## 6 · SUM

You now hold every booking from both systems in one table at one grain, with every rejected record listed. Record in `docs/decisions-log.md`: the repeated-header count, the city spelling rule, the reject count and reasons. Record in `docs/approach-log.md`: why booking grain and not booking-test grain for the headline, and what the alternative would have changed.

## Before you move on

- What is a booking? One sentence. Does a rescheduled nova record count as a second booking?
- How many bookings did the validator reject, and could any of them have been real visits?
- If a reviewer re-ran this on a fresh Codespace, would they get the same reject count? Why?

#### ✅ Answers: "Before you move on"

**1. What is a booking? One sentence. Does a rescheduled nova record count as a second booking?**

> **A booking is one visit (one scheduled sample collection) by one patient, carrying one or more tests.**

A rescheduled nova record is **not** a second booking. It's the same visit moved to a new slot. Nova writes a new record (pointing back via `rescheduled_from`) and marks the old one `rescheduled`. Legacy simply overwrote the slot. To make the systems comparable, each rescheduled pair must count **once**: keep the record with the slot that actually happened (the new one) and drop the superseded original. Notebook 03 drops the 1,499 originals. One more step is needed there, explained in 03: the collection is recorded against the **original** id, so it has to be carried over to the new record.

**2. How many bookings did the validator reject, and could any of them have been real visits?**

**3,107 bookings (0.55% of 567,986), all from legacy.** Yes, most of them were almost certainly real visits:
- **1,027 "slot before creation"**: creation is *exactly* 2 days after the slot every time. That's a systematic timestamp error in the vendor's `created_at`, while the slot (the visit itself) is probably right. A real visit with a bad booking timestamp.
- **793 malformed ids**: a valid id with an extra `0` appended. The trimmed ids don't collide with anything, so these are real, unique bookings with a corrupted key.
- **1,287 empty test codes**: the booking happened, but at least one test can't be identified or priced.

So rejecting them **undercounts** visits by about 0.55%. Because the rejects are spread evenly (1,074 in H1 2025 and 1,004 in H1 2026), the effect on the *growth rate* is tiny: with them included it would be +5.78% raw instead of +5.84% after rejects on the raw-row basis. But it's a decision to log with its alternatives: (a) reject, as done now, (b) repair the 2-day shift and the trailing zero as logged reversible rules, (c) keep the booking and drop only the unpriceable test line.

**3. If a reviewer re-ran this on a fresh Codespace, would they get the same reject count? Why?**

**Yes, 3,107**, provided the inputs and library versions are the same, because every step is deterministic:
- the input files are fixed and verified (MANIFEST SHA-256 check in 01),
- files are read in **sorted** order, and `read_many` puts results back **in path order** regardless of which worker finishes first,
- the rules are pure functions with no randomness, clock or network,
- `validate_many` is a plain loop with no sampling.

**What could change it:** different **pydantic** or **pandas** versions (pandas 3 changes default string dtypes, and pydantic coercion rules have changed between versions). So pin versions in `requirements.txt`. Replacing `data/raw` with a different pack would also change it, which is why the MANIFEST check comes first.